In [1]:
import torch

In [5]:
def prepare_sparse_tensor(batch_points, voxel_size=0.05):
    """
    batch_points: Tensor of shape (B, N, 6)
    Returns:
        coords: (B*N', 4) [batch_idx, x, y, z]
        feats:  (B*N', C)
    """
    B, N, _ = batch_points.shape
    all_coords = []
    all_feats = []
    
    for b in range(B):
        pc = batch_points[b]  # (N, 6)
        xyz = pc[:, :3]
        feat = pc[:, 3:]

        # Voxelize coordinates
        coords = torch.floor(xyz / voxel_size).int()
        batch_idx = torch.full((coords.shape[0], 1), b, dtype=torch.int)
        coords_batched = torch.cat([batch_idx, coords], dim=1)  # (N, 4)

        all_coords.append(coords_batched)
        all_feats.append(feat)

    coords = torch.cat(all_coords, dim=0)  # (B*N, 4)
    feats = torch.cat(all_feats, dim=0)    # (B*N, C)
    return coords, feats

In [6]:
import torch
import MinkowskiEngine as ME
import torch.nn as nn

class MinkowskiPointNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=128):
        super().__init__()
        self.projector = ME.MinkowskiLinear(in_channels, out_channels)
        self.pool = ME.MinkowskiGlobalAvgPooling()

    def forward(self, coords, feats):
        # coords: (N, 4), feats: (N, C)
        x = ME.SparseTensor(features=feats, coordinates=coords)
        x = self.projector(x)
        x = self.pool(x)  # Outputs a SparseTensor with shape (B, out_channels)
        return x.F  # Return dense feature: (B, out_channels)

In [7]:
B = 8
N = 40000
pointcloud = torch.randn(B, N, 6)  # Your input

coords, feats = prepare_sparse_tensor(pointcloud)

model = MinkowskiPointNet(in_channels=3, out_channels=128)
embedding = model(coords, feats)  # (B, 128)

print("Final embedding shape:", embedding.shape)

Final embedding shape: torch.Size([8, 128])
